# Kenya Coffee Research - Statistical Analysis
## Predicting Coffee Yield Variation Using Climate Data (1990-2024)

**Author:** Doreen Molly Wanjiru
**Dataset:** 35 years (1990-2024)
**Date:** June 2026

---

## Analysis Workflow

This notebook follows the following steps:
1. **Descriptive Statistics** - Understanding the data
2. **Exploratory Visualizations** - Identifying patterns
3. **Regression Assumptions Testing** - Validating modeling approach
4. **Regression Modeling** - Identifying significant variables
5. **Advanced Visualizations** - Communicating findings

---

##Setup & Data Loading

In [2]:
import sys
!{sys.executable} -m pip install scipy

print("SciPy installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 14.0 MB/s  0:00:01m0:00:0100:01
SciPy installed successfully!


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime

In [5]:
# setting up display and visualization settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("="*80)
print("KENYA COFFEE RESEARCH - STATISTICAL ANALYSIS")
print("="*80)
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("="*80)

KENYA COFFEE RESEARCH - STATISTICAL ANALYSIS
Analysis Date: 2026-06-23 12:02


In [7]:
data_path = '/Users/dwanjiru/projects/kenya-coffee-research/data/processed/master_data_1990_2024.csv'
df = pd.read_csv(data_path)

print(f"\n Data loaded successfully!")
print(f"  Shape: {df.shape[0]} observations × {df.shape[1]} variables")
print(f"  Time period: {df['Year'].min()}-{df['Year'].max()}")
print(f"\nVariables in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")


 Data loaded successfully!
  Shape: 35 observations × 10 variables
  Time period: 1990-2024

Variables in dataset:
  1. Year
  2. Area_Ha
  3. Production_Tonnes
  4. Yield_Kg_Ha
  5. Production_Value_1000USD
  6. Value_Unit
  7. Temperature
  8. Precipitation
  9. Humidity
  10. Solar_Radiation


In [8]:
print("\nFirst 10 rows:")
df.head(10)


First 10 rows:


,Year,Area_Ha,Production_Tonnes,Yield_Kg_Ha,Production_Value_1000USD,Value_Unit,Temperature,Precipitation,Humidity,Solar_Radiation
0,1990,153100.000,103900.000,678.600,425834,1000 USD,17.825,44.499,11.482,20.186
1,1991,155400.000,86400.000,556.000,354110,1000 USD,18.321,36.425,10.948,20.752
2,1992,153800.000,85300.000,554.600,349602,1000 USD,18.271,35.837,10.859,20.538
3,1993,158200.000,75100.000,474.700,307797,1000 USD,17.989,38.087,10.878,20.632
4,1994,158700.000,79900.000,503.500,327470,1000 USD,18.279,44.731,11.013,21.080
5,1995,160500.000,95400.000,594.400,390996,1000 USD,17.979,41.650,11.287,20.509
6,1996,177400.000,97976.000,552.300,401554,1000 USD,17.902,39.834,11.150,20.241
7,1997,176907.000,68642.000,388.000,281329,1000 USD,18.290,49.452,11.041,20.479
8,1998,178500.000,53715.000,300.900,220151,1000 USD,17.919,43.059,11.916,20.074
9,1999,170000.000,68100.000,400.600,279108,1000 USD,18.631,28.574,10.454,20.846


# Section 1: Descritpive Statistics

We will understand our data through:
1. **Summary statistics** (mean, median, SD, min, max)
2. **Distribution characteristics** (skewness, kurtosis)
3. **Variability measures** (coefficient of variation)
4. **Normality tests** (Shapiro-Wilk)
5. **Bivariate correlations** (relationships between variables)
6. **Temporal trends** (changes over time)

---
## 1.1: Defining Variable Groups

In [9]:
dependent_var = 'Yield_Kg_Ha'
independent_vars = ['Temperature', 'Precipitation', 'Humidity', 'Solar_Radiation']
all_vars = [dependent_var] + independent_vars

print("Variable Structure:")
print(f"  Dependent variable (Y):     {dependent_var}")
print(f"  Independent variables (X):  {', '.join(independent_vars)}")

Variable Structure:
  Dependent variable (Y):     Yield_Kg_Ha
  Independent variables (X):  Temperature, Precipitation, Humidity, Solar_Radiation


## 1.2 Summary Statistics
Let's start with basic descriptive statistics for all key variables.

In [11]:
summary = df[all_vars].describe()

print("="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(summary.round(2))

print("\n" + "="*80)
print("ADDITIONAL STATISTICS")
print("="*80)
print(f"{'Variable':<20} {'Median':>10} {'Range':>12} {'IQR':>10}")
print("-"*52)

for var in all_vars:
    median = df[var].median()
    range_val = df[var].max() - df[var].min()
    iqr = df[var].quantile(0.75) - df[var].quantile(0.25)
    print(f"{var:<20} {median:>10.2f} {range_val:>12.2f} {iqr:>10.2f}")

SUMMARY STATISTICS
       Yield_Kg_Ha  Temperature  Precipitation  Humidity  Solar_Radiation
count       35.000       35.000         35.000    35.000           35.000
mean       398.610       18.480         40.480    11.260           20.580
std        109.410        0.380         10.030     0.480            0.370
min        262.500       17.820         22.300    10.130           20.010
25%        311.150       18.280         35.990    10.990           20.280
50%        370.000       18.460         39.830    11.180           20.540
75%        462.200       18.760         44.620    11.500           20.840
max        678.600       19.200         66.520    12.580           21.320

ADDITIONAL STATISTICS
Variable                 Median        Range        IQR
----------------------------------------------------
Yield_Kg_Ha              370.00       416.10     151.05
Temperature               18.46         1.38       0.48
Precipitation             39.83        44.22       8.63
Humidity       

#### Intepretation:

Based on the data, we can see that our dependent variable(yield) ranges between 262.5 kg/ha and 678.6 kg/ha (a 2.6x difference). Yield variation indicates environmental factors strongly influence crop performance.The mean(398.61) is also slightly right-skewed since it is greater than the median(370.00). 

Precipitation is the independent variable that has the widest range: 22.3 - 66.5 mm (nearly a 3x difference). It will probably be the strongest predictor of yield due to its high variability. 

Temperature, humidity and solar radiation on the other hand have limited variations hence have limited standalone predictive power. 

## 1.3 Distribution Characteristics

**Skewness** to measure asymmetry:
- **0** = perfectly symmetric
- **-ve** = left-skewed
- **+ve** = right-skewed
- |skewness| < 0.5 is fairly symmetric

**Kurtosis** to measure tail heaviness:
- **0** = normal distribution
- **+ve** = heavy tails (more outliers)
- **-ve** = light tails (fewer outliers)


In [16]:
print("="*80)
print("DISTRIBUTION CHARACTERISTICS")
print("="*80)
print(f"{'Variable':<20} {'Mean':>10} {'Std Dev':>10} {'Skewness':>10} {'Kurtosis':>10}")
print("-"*65)

for var in all_vars:
    mean = df[var].mean()
    std = df[var].std()
    skew = stats.skew(df[var])
    kurt = stats.kurtosis(df[var])
    print(f"{var:<20} {mean:>10.2f} {std:>10.2f} {skew:>10.3f} {kurt:>10.3f}")

print("\nInterpretation Guide:")
print("  Skewness: |value| < 0.5 (symmetric), 0.5-1 (moderate), >1 (highly skewed)")
print("  Kurtosis: 0 (normal), >0 (heavy-tailed), <0 (light-tailed)")

DISTRIBUTION CHARACTERISTICS
Variable                   Mean    Std Dev   Skewness   Kurtosis
-----------------------------------------------------------------
Yield_Kg_Ha              398.61     109.41      0.791     -0.308
Temperature               18.48       0.38     -0.020     -0.812
Precipitation             40.48      10.03      0.704      0.442
Humidity                  11.26       0.48      0.477      0.946
Solar_Radiation           20.58       0.37      0.380     -0.669

Interpretation Guide:
  Skewness: |value| < 0.5 (symmetric), 0.5-1 (moderate), >1 (highly skewed)
  Kurtosis: 0 (normal), >0 (heavy-tailed), <0 (light-tailed)


Yield_Kg_Ha:
Distribution has a longer tail toward high yields(skewness = 0.791); most observations cluster at lower-to-average values with some high performers.
Fewer extreme values than normal distribution (Kurtosis = -0.308); yields are relatively concentrated around the mean without many outliers.
Variable may benefit from log transformation if used in linear regression to normalize distribution.

Precipitation:
Most observations (skewness = 0.704) are at lower-to-moderate precipitation levels with some notably high rainfall events.
Modest presence of extreme precipitation values(Kurtosis = 0.442); occasional heavy rainfall periods.
Asymmetric distribution. May consider square root or log transformation to normalize for certain models.

Humidity:
Nearly symmetric with values evenly distributed around the mean.
Heavy-tailed (kurtosis = 0.946); more extreme values than normal distribution despite narrow range.
Despite low variance, frequent extremes may indicate threshold effects worth investigating.

Temperature and Solar Radiation:
Both are symmetric, with values evenly distributed around the mean. They have more uniform distribution without many extremes. 






## 1.4 Coefficient of Variation (CV)

The CV measures relative variability (standard deviation as a percentage of the mean).

**Interpretation:**
- **CV < 15%** = Low variability (stable)
- **CV 15-30%** = Moderate variability
- **CV > 30%** = High variability (volatile)

This is useful for comparing variability across variables with different units.

In [17]:
print("="*80)
print("COEFFICIENT OF VARIATION (CV)")
print("="*80)
print(f"{'Variable':<20} {'Mean':>10} {'Std Dev':>10} {'CV %':>10} {'Interpretation':>15}")
print("-"*75)

for var in all_vars:
    mean = df[var].mean()
    std = df[var].std()
    cv = (std/mean) * 100

    if cv < 15:
        interp = "Low"
    elif cv < 30:
        interp = "Moderate"
    else:
        interp = "High"
    
    print(f"{var:<20} {mean:>10.2f} {std:>10.2f} {cv:>10.2f} {interp:>15}")


COEFFICIENT OF VARIATION (CV)
Variable                   Mean    Std Dev       CV %  Interpretation
---------------------------------------------------------------------------
Yield_Kg_Ha              398.61     109.41      27.45        Moderate
Temperature               18.48       0.38       2.04             Low
Precipitation             40.48      10.03      24.78        Moderate
Humidity                  11.26       0.48       4.30             Low
Solar_Radiation           20.58       0.37       1.78             Low


## 1.5 Normality Tests

We check if variables follow a normal distribution.

**Shapiro-Wilk Test:**
- **H₀** (null hypothesis): Data is normally distributed
- **H₁** (alternative): Data is NOT normally distributed
- **Decision rule**: If p-value > 0.05, we accept H₀ (data is normal)


In [18]:
print("="*80)
print("NORMALITY TESTS (Shapiro-Wilk)")
print("="*80)
print(f"{'Variable':<20} {'W-statistic':>12} {'p-value':>10} {'Normal?':>12}")
print("-"*54)

for var in all_vars:
    stat, p_value = stats.shapiro(df[var])
    is_normal = "Yes ✓" if p_value > 0.05 else "No ✗"
    print(f"{var:<20} {stat:>12.4f} {p_value:>10.4f} {is_normal:>12}")

print("\nInterpretation:")
print("  If p > 0.05: Variable is normally distributed ✓")
print("  If p < 0.05: Variable deviates from normality ✗")

NORMALITY TESTS (Shapiro-Wilk)
Variable              W-statistic    p-value      Normal?
------------------------------------------------------
Yield_Kg_Ha                0.9165     0.0113         No ✗
Temperature                0.9684     0.4003        Yes ✓
Precipitation              0.9428     0.0682        Yes ✓
Humidity                   0.9670     0.3651        Yes ✓
Solar_Radiation            0.9589     0.2113        Yes ✓

Interpretation:
  If p > 0.05: Variable is normally distributed ✓
  If p < 0.05: Variable deviates from normality ✗


Shapiro-Wilk tests reveal that Yield_Kg_Ha significantly deviates from normality, which is consistent with its moderate right skewness (0.791). All predictor variables pass normality tests. 

The non-normality of the response variable suggests potential need for transformation (log or square root) to improve model performance and meet regression assumptions. 

In [19]:
residuals = model.resid  # actual - predicted

from scipy import stats
stat, p = stats.shapiro(residuals)
print(f"Residual normality test: p = {p:.4f}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residuals')

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Residual Q-Q Plot')
plt.tight_layout()
plt.show()

NameError: name 'model' is not defined